# Synthetic Streaming-Platform Data

A reproducible, simulation-based example that generates related catalog, user, and listening-event tables without using individual-level source records.

## tl;dr

With the default seed and scenario parameters, the generator produces:

- 360 unique tracks from 10 artists, 3 albums per artist, and 12 tracks per album;
- 1,000 synthetic users sampled across Brazil's 27 federative units using aggregate 2021 population weights;
- 2,992 paired listening events across 100 half-hour periods;
- zero duplicate primary keys, missing event values, or broken user/track foreign keys.

These are simulated records, not estimates of real platform behavior. Synthetic data is also not automatically anonymous when a generator is trained on sensitive microdata; this repository avoids that issue by using explicit simulation assumptions and public aggregates only.

## Context & Methods

The study models a normalized analytical system with four tables:

1. `state_population`: public aggregate state populations and sampling weights;
2. `catalog`: synthetic artists, albums, tracks, and latent popularity weights;
3. `users`: deterministic identifiers and scenario-based attributes;
4. `events`: paired user–track interactions generated at regular time ticks.

A single seeded `numpy.random.Generator` controls every random draw. This removes the hidden nondeterminism caused by mixing Python's `random` module, NumPy's global state, and process-dependent `hash()` identifiers.

### Key assumptions

- State sampling follows the relative population totals in the included 2021 IBGE municipal estimates.
- Age and sex-state distributions are scenario parameters, not empirical demographic estimates.
- Track popularity follows a Dirichlet draw and remains fixed during this introductory simulation.
- Event counts per period follow a Poisson distribution.
- Users are sampled uniformly after the synthetic user table has been created.
- Listening duration is sampled uniformly from 30 to 300 seconds.

Changing these assumptions changes the synthetic population. Fidelity must therefore be evaluated against the intended use case, not judged from plausible-looking rows.

## Data

The only external input is `streaming_platform/estimativa_dou_2021.csv`, a semicolon-delimited table with population estimates for 5,570 Brazilian municipalities. The generator aggregates this public table to 27 federative units. It does not ingest names, account histories, device identifiers, or other personal records.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    raise FileNotFoundError("Run this notebook from the repository root.")

sys.path.insert(0, str(ROOT / "src"))

from synthetic_data import StreamingConfig, StreamingPlatformGenerator

sns.set_theme(style="whitegrid")

## Results

### 1. Configure and run the simulation

In [ ]:
config = StreamingConfig(
    seed=42,
    n_artists=10,
    albums_per_artist=3,
    tracks_per_album=12,
    n_users=1_000,
    start="2021-01-01",
    periods=100,
    frequency="30min",
    mean_events_per_period=30,
)

population_path = ROOT / "streaming_platform" / "estimativa_dou_2021.csv"
generator = StreamingPlatformGenerator(config)
tables = generator.generate(population_path)

state_population = tables["state_population"]
catalog = tables["catalog"]
users = tables["users"]
events = tables["events"]

### 2. Inspect bounded samples

In [ ]:
print("State population")
display(state_population.head())

print("Catalog")
display(catalog.head())

print("Users")
display(users.head())

print("Events")
display(events.head())

### 3. Validate relational integrity

In [ ]:
quality = generator.quality_report(tables)
quality.to_frame()

In [ ]:
assert quality["users"] == config.n_users
assert quality["tracks"] == (
    config.n_artists
    * config.albums_per_artist
    * config.tracks_per_album
)
assert quality["events"] == 2_992
assert quality[
    [
        "duplicate_user_ids",
        "duplicate_track_ids",
        "duplicate_event_ids",
        "unknown_event_users",
        "unknown_event_tracks",
        "missing_event_values",
    ]
].eq(0).all()
assert state_population["sampling_probability"].sum().round(12) == 1.0

print("All primary-key, foreign-key, missingness, and probability checks passed.")

### 4. Verify reproducibility

A new generator with the same configuration must reproduce every table exactly.

In [ ]:
replicated_tables = StreamingPlatformGenerator(config).generate(population_path)

for table_name in tables:
    pd.testing.assert_frame_equal(
        tables[table_name],
        replicated_tables[table_name],
    )

print("The complete simulation is reproducible for seed 42.")

### 5. Compare target and realized state shares

Sampling variability is expected because only 1,000 users are generated. The plot compares the aggregate population weights with the realized synthetic-user proportions.

In [ ]:
realized_state_share = (
    users["uf"]
    .value_counts(normalize=True)
    .rename("realized_share")
    .rename_axis("uf")
    .reset_index()
)

state_comparison = state_population.merge(realized_state_share, on="uf")
state_comparison["absolute_error"] = (
    state_comparison["realized_share"]
    - state_comparison["sampling_probability"]
).abs()

plot_data = (
    state_comparison.nlargest(10, "sampling_probability")
    .melt(
        id_vars=["uf"],
        value_vars=["sampling_probability", "realized_share"],
        var_name="series",
        value_name="share",
    )
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=plot_data, x="uf", y="share", hue="series", ax=ax)
ax.set_title("Expected and realized shares for the ten largest state weights")
ax.set_xlabel("Federative unit")
ax.set_ylabel("Share")
plt.tight_layout()

### 6. Check whether simulated events reflect track weights

In [ ]:
realized_track_share = (
    events["track_id"]
    .value_counts(normalize=True)
    .rename("realized_event_share")
    .rename_axis("track_id")
    .reset_index()
)
track_comparison = catalog.merge(realized_track_share, on="track_id", how="left")
track_comparison["realized_event_share"] = track_comparison[
    "realized_event_share"
].fillna(0.0)

popularity_correlation = track_comparison[
    ["sampling_probability", "realized_event_share"]
].corr().iloc[0, 1]

print(f"Correlation between configured and realized track shares: {popularity_correlation:.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(
    data=track_comparison,
    x="sampling_probability",
    y="realized_event_share",
    alpha=0.65,
    ax=ax,
)
ax.set_title("Configured track weights versus realized event shares")
ax.set_xlabel("Configured sampling probability")
ax.set_ylabel("Realized event share")
plt.tight_layout()

## Takeaways

The refactored generator produces normalized tables with deterministic identifiers and verified referential integrity. Its explicit parameters make assumptions inspectable, while the seeded random-number generator makes failures reproducible. Population weighting is grounded in a public aggregate source; no arbitrary state-specific behavioral prior is imposed.

The simulation remains deliberately simple. It does not model seasonality, recommendation feedback, artist releases, churn, household structure, or changing track popularity. Those mechanisms should be introduced as separately testable components rather than hidden inside loosely coupled notebook functions.

Synthetic data quality has at least three distinct dimensions: statistical fidelity, analytical utility, and disclosure risk. Passing schema and integrity checks addresses only the first layer of engineering correctness. It does not establish equivalence to a real platform population, fairness across groups, or privacy protection.

## Reuse

The implementation is available as `StreamingPlatformGenerator` in `src/synthetic_data/streaming.py`. See the repository README for a minimal Python example and [REFERENCES.md](../REFERENCES.md) for the methodological bibliography.